# 【課題2】2024年版ものづくり白書（全340ページ・31MB）を対象としたRAG構築

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

本ノートブックでは、**「2024年版ものづくり白書（令和5年度ものづくり基盤技術の振興施策）」**を題材に、検索精度を高めるための3段階アプローチを実装・検証します。

### 🎯 本ノートブックの学習目標
1. **大容量白書のチャンキング設計**: 1節単位（数千〜数万文字）の肥大化を防ぎ、**章・節メタデータを付与しながら600〜800文字に分割**する技法
2. **段階的な検索エンジンの実装**:
   - **① キーワード検索 (BM25)**: 「価格転嫁」「GX」などの固有名詞・専門用語をピンポイント検索
   - **② ベクトル検索 (ChromaDB + gemini-embedding-001)**: 類義語やニュアンスでの意味的検索（MRL対応で768次元に最適化）
   - **③ ハイブリッド検索 (RRF: Reciprocal Rank Fusion)**: ①と②の順位を融合して最高精度を実現
3. **出典（章・節・ページ番号）付き回答生成**: Gemini 2.5 Flashによる根拠付きQA
4. **Gradio Web UI**: Colabセル内での操作画面と共有URLの発行

## 1. 必要なライブラリのインストール
キーワード検索用の `rank-bm25` を含む必要ライブラリをインストールします。

In [ ]:
!pip install -q \
    langchain==0.3.14 \
    langchain-community==0.3.14 \
    langchain-google-genai==2.0.8 \
    chromadb==0.6.3 \
    pypdf==5.2.0 \
    gradio==5.12.0 \
    pandas==2.2.3 \
    rank-bm25==0.2.2 \
    google-genai==2.4.0

print("✅ ライブラリのインストールが完了しました。")

## 2. Gemini API キーの設定
Colab画面左側の 🔑「シークレット（Secrets）」に `GEMINI_API_KEY` を登録し、「ノートブックへのアクセス」をONにしてください。

In [ ]:
import os
from google.colab import userdata

try:
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
    print("✅ GEMINI_API_KEY をColabシークレットから正常に設定しました。")
except Exception:
    from getpass import getpass
    os.environ["GEMINI_API_KEY"] = getpass("Gemini API Key を入力してください: ")

## 3. 白書PDF（31MB）の配置
経産省の公式サイトからダウンロードするか、GitHubにコミットしたPDFを配置します。

In [ ]:
import os
os.makedirs("data", exist_ok=True)
pdf_target = "data/document.pdf"

# まだファイルがない場合は経産省サイトまたは手動アップロードを案内
if not os.path.exists(pdf_target):
    print(f"[*] {pdf_target} を確認中...")
    print("※ 左メニューの📁ファイルから31MBの白書PDFをドラッグ＆ドロップして 'data/document.pdf' に配置してください。")
    print("※ または以下のコマンドで経産省サイトから直接ダウンロードできます:")
    print("!wget https://www.meti.go.jp/report/whitepaper/mono/2024/honbun/pdf/all.pdf -O data/document.pdf")
else:
    size_mb = os.path.getsize(pdf_target) / (1024*1024)
    print(f"✅ 白書PDFを確認しました: {pdf_target} ({size_mb:.2f} MB)")

## 4. 階層メタデータ（章・節・ページ）付きチャンク分割 & pandas集計
「1節まるごとだと長すぎる」という問題を解決するため、**章・節・見出しをメタデータとして引き継ぎながら、600〜800文字単位に分割**します。

In [ ]:
import re
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def extract_headings(text, current):
    meta = current.copy()
    p_match = re.search(r'(第[一二三四五六七八九十\d]+部\s+[^\n]{2,30})', text)
    if p_match: meta['part'] = p_match.group(1).strip()
    c_match = re.search(r'(第[一二三四五六七八九十\d]+章\s+[^\n]{2,35})', text)
    if c_match: meta['chapter'] = c_match.group(1).strip()
    s_match = re.search(r'(第[一二三四五六七八九十\d]+節\s+[^\n]{2,35})', text)
    if s_match: meta['section'] = s_match.group(1).strip()
    return meta

if os.path.exists(pdf_target):
    loader = PyPDFLoader(pdf_target)
    pages = loader.load()
    print(f"📖 読込ページ数: {len(pages)} ページ")
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=150,
        separators=["\n\n", "\n", "。", "、", " ", ""]
    )
    
    chunks = []
    records = []
    curr_meta = {'part': '第1部', 'chapter': '第1章', 'section': '第1節'}
    chunk_id = 0
    
    for p_doc in pages:
        p_num = p_doc.metadata.get('page', 0) + 1
        curr_meta = extract_headings(p_doc.page_content, curr_meta)
        p_chunks = splitter.split_text(p_doc.page_content)
        
        for txt in p_chunks:
            if len(txt.strip()) < 30: continue
            chunk_id += 1
            meta = {
                'chunk_id': chunk_id,
                'page': p_num,
                'part': curr_meta['part'],
                'chapter': curr_meta['chapter'],
                'section': curr_meta['section']
            }
            chunks.append(Document(page_content=txt.strip(), metadata=meta))
            records.append({
                'chunk_id': chunk_id,
                'page': p_num,
                'section': curr_meta['section'],
                'char_count': len(txt.strip()),
                'preview': txt.strip()[:60].replace('\n', ' ') + '...'
            })
            
    df_chunks = pd.DataFrame(records)
    print(f"✂️ 分割チャンク数: {len(chunks)} 件")
    print("\n📊 --- チャンクの文字数要約統計 (pandas describe) ---")
    display(df_chunks['char_count'].describe())
    print("\n📋 --- 検出された節ごとのチャンク数 (Top 5) ---")
    display(df_chunks['section'].value_counts().head(5))
else:
    print("※ PDF配置後にこのセルを実行してください。")

## 5-1. 【ステップ1】キーワード検索 (BM25) の実装
日本語トークナイズ（単語＋2-gram）とBM25アルゴリズムを用いて、専門用語の完全一致検索を実装します。

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize_jp(text):
    clean = text.lower()
    words = re.findall(r'[a-zA-Z0-9]+|[一-龥]{2,}|[ァ-ンヴー]{2,}', clean)
    bigrams = [clean[i:i+2] for i in range(len(clean)-1) if not re.match(r'\s', clean[i:i+2])]
    return list(set(words + bigrams)) or [clean]

if 'chunks' in locals() and len(chunks) > 0:
    print("[*] BM25インデックスを作成中...")
    corpus_tokens = [tokenize_jp(c.page_content) for c in chunks]
    bm25 = BM25Okapi(corpus_tokens)
    print("✅ BM25キーワード検索エンジンの準備が完了しました。")
    
    # 単体検索テスト
    test_kw = "価格転嫁"
    scores = bm25.get_scores(tokenize_jp(test_kw))
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:3]
    print(f"\n🔎 『{test_kw}』のBM25検索結果 (Top 3):")
    for r, idx in enumerate(top_idx, 1):
        m = chunks[idx].metadata
        print(f"  {r}. {m['section']} (p.{m['page']}) スコア: {scores[idx]:.2f}")
        print(f"     {chunks[idx].page_content[:80]}...")

## 5-2. 【ステップ2】ベクトル検索 (ChromaDB + gemini-embedding-001) の実装
Googleの最新多言語テキスト埋め込みモデル `models/gemini-embedding-001` を使用します。
MRL（マトリョーシカ表現学習）により `output_dimensionality=768` を指定することで、精度を保ちながらColabのメモリ消費を抑制し高速化します。

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

# 最新モデル: models/gemini-embedding-001 (MRL 768次元指定)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.environ["GEMINI_API_KEY"],
    output_dimensionality=768
)

if 'chunks' in locals() and len(chunks) > 0:
    print("[*] ChromaDB にベクトルインデックスを作成中 (gemini-embedding-001 / 768次元)...針")
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory="./chroma_db",
        collection_name="whitepaper_vectors"
    )
    print("✅ ベクトル検索エンジンの準備が完了しました。")
    
    # 単体検索テスト（言い換え表現で検索）
    test_query = "原材料費の高騰を取引価格に反映できているか？"
    vec_results = vector_store.similarity_search_with_relevance_scores(test_query, k=3)
    print(f"\n🔎 『{test_query}』のベクトル検索結果 (Top 3):")
    for r, (doc, sc) in enumerate(vec_results, 1):
        m = doc.metadata
        print(f"  {r}. {m['section']} (p.{m['page']}) 類似度: {sc:.3f}")
        print(f"     {doc.page_content[:80]}...")

## 5-3. 【ステップ3】ハイブリッド検索 (Reciprocal Rank Fusion: RRF) の実装
BM25の順位とベクトルの順位をRRF式で融合し、専門用語と意味理解の双方で優れた総合ランキングを算出します。

In [ ]:
def hybrid_search(query, top_k=4, rrf_k=60, w_kw=0.5, w_vec=0.5):
    # 1. BM25 検索
    kw_scores = bm25.get_scores(tokenize_jp(query))
    kw_top = sorted(range(len(kw_scores)), key=lambda i: kw_scores[i], reverse=True)[:top_k*2]
    
    # 2. ベクトル検索
    vec_res = vector_store.similarity_search_with_relevance_scores(query, k=top_k*2)
    
    fused = {}
    # BM25 RRF加算
    for rank, idx in enumerate(kw_top, 1):
        if kw_scores[idx] <= 0: continue
        c_id = chunks[idx].metadata['chunk_id']
        fused[c_id] = {
            'doc': chunks[idx],
            'score': w_kw * (1.0 / (rrf_k + rank)),
            'kw_rank': rank,
            'vec_rank': None
        }
        
    # ベクトル RRF加算
    for rank, (doc, sc) in enumerate(vec_res, 1):
        c_id = doc.metadata['chunk_id']
        score_val = w_vec * (1.0 / (rrf_k + rank))
        if c_id in fused:
            fused[c_id]['score'] += score_val
            fused[c_id]['vec_rank'] = rank
        else:
            fused[c_id] = {'doc': doc, 'score': score_val, 'kw_rank': None, 'vec_rank': rank}
            
    ranked = sorted(fused.values(), key=lambda x: x['score'], reverse=True)[:top_k]
    return ranked

if 'vector_store' in locals():
    compare_q = "ものづくり企業のGX・脱炭素化の取り組み"
    print(f"\n🏆 『{compare_q}』のハイブリッド検索 (Top 3):")
    for i, item in enumerate(hybrid_search(compare_q, top_k=3), 1):
        m = item['doc'].metadata
        print(f"  {i}. {m['section']} (p.{m['page']}) RRFスコア: {item['score']*100:.2f} (BM25順位:{item['kw_rank']}, ベクトル順位:{item['vec_rank']})")
        print(f"     {item['doc'].page_content[:80]}...")

## 6. Gemini 2.5 Flash による出典（章・節・ページ）付き回答生成
ハイブリッド検索結果をプロンプトに挿入し、根拠のある回答を生成します。

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)

prompt_tmpl = """あなたは「ものづくり白書」の専門アシスタントです。
以下の【資料】のみを根拠にして質問に回答してください。
資料に記載のない内容は推測せず「資料に記載がありません」と答え、回答文中に参照箇所の [第X章 第X節 p.Y] を必ず明記してください。

【資料】:
{context}

【質問】:
{question}

【回答】:"""

def ask_whitepaper(question, mode='hybrid'):
    hits = hybrid_search(question, top_k=4)
    context_parts = []
    for i, h in enumerate(hits, 1):
        m = h['doc'].metadata
        context_parts.append(f"--- [資料{i}: {m['chapter']} {m['section']} (p.{m['page']})] ---\n{h['doc'].page_content}")
    context_text = "\n\n".join(context_parts)
    
    chain = PromptTemplate.from_template(prompt_tmpl) | llm | StrOutputParser()
    return chain.invoke({"context": context_text, "question": question}), hits

if 'vector_store' in locals():
    sample_q = "中小製造業における価格転嫁の現状と直面している課題について教えてください。"
    print(f"❓ 質問: {sample_q}\n")
    answer, used_hits = ask_whitepaper(sample_q)
    print(f"💡 回答:\n{answer}")

## 7. Gradio Web チャットボットUI（検索方式切替付き）の起動
3つの検索エンジンを自由に切り替えながら対話テストできるUIを起動します。

In [ ]:
import gradio as gr

def chat_handler(message, history, search_mode):
    if 'vector_store' not in globals() or 'bm25' not in globals():
        return history + [(message, "⚠️ 先にインデックス作成セルを実行してください。")]
    
    # 選択されたモードで検索
    if search_mode == "キーワード検索 (BM25)":
        kw_scores = bm25.get_scores(tokenize_jp(message))
        top_idx = sorted(range(len(kw_scores)), key=lambda i: kw_scores[i], reverse=True)[:4]
        hits = [{'doc': chunks[i], 'kw_rank': r, 'vec_rank': None} for r, i in enumerate(top_idx, 1)]
    elif search_mode == "ベクトル検索 (ChromaDB)":
        vec_res = vector_store.similarity_search_with_relevance_scores(message, k=4)
        hits = [{'doc': doc, 'kw_rank': None, 'vec_rank': r} for r, (doc, _) in enumerate(vec_res, 1)]
    else:
        hits = hybrid_search(message, top_k=4)
        
    context_parts = []
    sources_line = []
    for i, h in enumerate(hits, 1):
        m = h['doc'].metadata
        loc = f"{m.get('chapter', '')} {m.get('section', '')} [p.{m.get('page')}]"
        context_parts.append(f"--- [資料{i}: {loc}] ---\n{h['doc'].page_content}")
        sources_line.append(loc)
        
    chain = PromptTemplate.from_template(prompt_tmpl) | llm | StrOutputParser()
    reply = chain.invoke({"context": "\n\n".join(context_parts), "question": message})
    
    full_reply = reply + f"\n\n---\n**📚 参照した資料 ({search_mode}):**\n" + "\n".join([f"- {s}" for s in sources_line])
    history.append((message, full_reply))
    return history

with gr.Blocks(title="2024年版ものづくり白書 RAG") as demo:
    gr.Markdown("# 📘 2024年版ものづくり白書 RAG チャットボット")
    gr.Markdown("**3段階検索切替: ① キーワード検索(BM25) / ② ベクトル検索 / ③ ハイブリッド検索(RRF)**")
    
    mode_select = gr.Radio(
        choices=["ハイブリッド検索 (RRF統合) [推奨]", "キーワード検索 (BM25)", "ベクトル検索 (ChromaDB)"],
        value="ハイブリッド検索 (RRF統合) [推奨]",
        label="検索方式"
    )
    chatbot_ui = gr.Chatbot(height=450)
    msg = gr.Textbox(placeholder="質問を入力 (例: 価格転嫁の進捗状況と課題について教えてください)...", label="質問")
    
    msg.submit(chat_handler, [msg, chatbot_ui, mode_select], chatbot_ui)
    msg.submit(lambda: "", None, msg)

demo.launch(share=True, debug=True)